# Laboratorio 2 — Complejidad y búsqueda de hiperparámetros (AlpesHearth)

Este notebook está construido **siguiendo el estilo del Lab 1 entregado** (limpieza + preprocesamiento con `ColumnTransformer`) y las **actividades de práctica** (regresión lineal/polinomial/regularizada).  
Cubre explícitamente las actividades 1–7 del enunciado del Laboratorio 2.

**Entrega:** Notebook ejecutado + video (máx. 3 min).

## 0. Setup

In [ ]:
import pandas as pd
import numpy as np
import re

import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, KFold, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, MinMaxScaler, PolynomialFeatures
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 140)

RANDOM_STATE = 42
TEST_SIZE = 0.25

TARGET = "CVD Risk Score"
AUX_LABEL = "CVD Risk Level"   # etiqueta auxiliar (NO usar como feature para regresión)

## 1. Carga de datos (igual que en el Lab 1)

In [ ]:
def read_csv_auto_sep(path: str) -> pd.DataFrame:
    # Detecta separador por primera línea
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        first_line = f.readline()
    sep = ";" if first_line.count(";") > first_line.count(",") else ","
    return pd.read_csv(path, sep=sep)

train_path = "data/Datos Lab 1.csv"
test_path = "data/Datos Test Lab 1.csv"
dicc_path = "data/DiccPacientes.xlsx"

train_raw = read_csv_auto_sep(train_path)
test_raw  = read_csv_auto_sep(test_path)
dicc_raw  = pd.read_excel(dicc_path)

# Limpieza básica de nombres de columnas
train_raw.columns = train_raw.columns.str.strip()
test_raw.columns  = test_raw.columns.str.strip()
dicc_raw.columns  = dicc_raw.columns.str.strip()

print("Train shape:", train_raw.shape)
print("Test  shape:", test_raw.shape)
display(dicc_raw.head(10))

print("Columnas extra en train (vs test):", sorted(list(set(train_raw.columns) - set(test_raw.columns))))

## 2. Limpieza y normalización (basado en el Lab 1)

In [ ]:
CAT_COLS = [
    "Sex",
    "Smoking Status",
    "Diabetes Status",
    "Physical Activity Level",
    "Family History of CVD",
    "Blood Pressure Category",
]

def parse_bp_series(series: pd.Series):
    sys_vals, dia_vals = [], []
    for v in series.astype(str):
        m = re.match(r"^\s*(\d+)\s*/\s*(\d+)\s*$", v)
        if m:
            sys_vals.append(float(m.group(1)))
            dia_vals.append(float(m.group(2)))
        else:
            sys_vals.append(np.nan)
            dia_vals.append(np.nan)
    return pd.Series(sys_vals, index=series.index), pd.Series(dia_vals, index=series.index)

def normalize_categories(df: pd.DataFrame, cat_cols: list[str]) -> pd.DataFrame:
    df = df.copy()
    for c in cat_cols:
        if c in df.columns:
            df[c] = df[c].astype("string").str.strip()
    return df

def clean_common(df: pd.DataFrame, cat_cols: list[str] = None) -> pd.DataFrame:
    if cat_cols is None:
        cat_cols = CAT_COLS

    df = df.copy()
    df = normalize_categories(df, cat_cols)

    # Validez numérica mínima
    if "Estimated LDL (mg/dL)" in df.columns:
        df.loc[df["Estimated LDL (mg/dL)"] < 0, "Estimated LDL (mg/dL)"] = np.nan
    if "Total Cholesterol (mg/dL)" in df.columns:
        df.loc[df["Total Cholesterol (mg/dL)"] < 0, "Total Cholesterol (mg/dL)"] = np.nan

    # Completar BP numérica desde el string "120/80"
    if "Blood Pressure (mmHg)" in df.columns:
        sys_bp, dia_bp = parse_bp_series(df["Blood Pressure (mmHg)"])
        if "Systolic BP" in df.columns:
            df["Systolic BP"] = df["Systolic BP"].fillna(sys_bp)
        if "Diastolic BP" in df.columns:
            df["Diastolic BP"] = df["Diastolic BP"].fillna(dia_bp)

    return df

# Train: quitar duplicados y filas sin target
train = train_raw.drop_duplicates().dropna(subset=[TARGET]).copy()
train = clean_common(train)

# Test: quitar duplicados (si aplica) y limpiar
test = test_raw.drop_duplicates().copy()
test = clean_common(test)

print("Train limpio shape:", train.shape)
print("Test  limpio shape:", test.shape)

display(train.head())

## 3. Definición de features/target y split Train/Test (test congelado)

In [ ]:
# Columnas a remover siempre (igual que Lab 1):
DROP_ALWAYS = ["Patient ID", "Date of Service", AUX_LABEL, "Blood Pressure (mmHg)"]

X = train.drop(columns=[TARGET] + DROP_ALWAYS, errors="ignore").copy()
y = train[TARGET].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE
)

print("X_train:", X_train.shape, "X_test:", X_test.shape)

## 4. Preprocesamiento (ColumnTransformer)

In [ ]:
def build_preprocess_with_poly(
    X_sample: pd.DataFrame,
    scaler_numeric,
    degree: int
) -> ColumnTransformer:
    """Preproceso para modelos polinomiales:
    - num: imputación + (scaler) + PolynomialFeatures
    - cat: imputación + OneHot
    """
    num_cols = X_sample.select_dtypes(include=[np.number]).columns.tolist()
    cat_cols = [c for c in X_sample.columns if c not in num_cols]

    num_steps = [("imputer", SimpleImputer(strategy="median"))]
    num_steps.append(("scaler", scaler_numeric))
    num_steps.append(("poly", PolynomialFeatures(degree=degree, include_bias=False)))

    cat_steps = [
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", drop="first")),
    ]

    return ColumnTransformer(
        transformers=[
            ("num", Pipeline(num_steps), num_cols),
            ("cat", Pipeline(cat_steps), cat_cols),
        ],
        remainder="drop"
    )

def build_preprocess_no_poly(
    X_sample: pd.DataFrame,
    scaler_numeric
) -> ColumnTransformer:
    """Preproceso para Ridge/Lasso sin polinomios."""
    num_cols = X_sample.select_dtypes(include=[np.number]).columns.tolist()
    cat_cols = [c for c in X_sample.columns if c not in num_cols]

    num_steps = [("imputer", SimpleImputer(strategy="median"))]
    num_steps.append(("scaler", scaler_numeric))

    cat_steps = [
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", drop="first")),
    ]

    return ColumnTransformer(
        transformers=[
            ("num", Pipeline(num_steps), num_cols),
            ("cat", Pipeline(cat_steps), cat_cols),
        ],
        remainder="drop"
    )

def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

def metrics(y_true, y_pred):
    return {
        "RMSE": rmse(y_true, y_pred),
        "MAE": float(mean_absolute_error(y_true, y_pred)),
        "R2": float(r2_score(y_true, y_pred)),
    }

## 5. Configuración de Validación Cruzada + scoring

In [ ]:
cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

scoring = {
    "RMSE": "neg_root_mean_squared_error",
    "MAE": "neg_mean_absolute_error",
    "R2": "r2",
}

## Actividad 1 — Regresión polinomial (Pipeline + GridSearchCV)

In [ ]:
# Pipeline polinomial: el preprocesamiento también se parametriza en GridSearch
poly_pipe = Pipeline([
    ("preprocess", build_preprocess_with_poly(X_train, scaler_numeric=StandardScaler(), degree=2)),
    ("reg", LinearRegression()),
])

param_grid_poly = {
    # Reemplazamos todo el preprocesador por versiones con distinto scaler/degree.
    "preprocess": [
        build_preprocess_with_poly(X_train, scaler_numeric=StandardScaler(), degree=d) for d in [1,2,3,4,5]
    ] + [
        build_preprocess_with_poly(X_train, scaler_numeric=MinMaxScaler(), degree=d) for d in [1,2,3,4,5]
    ] + [
        build_preprocess_with_poly(X_train, scaler_numeric="passthrough", degree=d) for d in [1,2,3,4,5]
    ]
}

gs_poly = GridSearchCV(
    poly_pipe,
    param_grid=param_grid_poly,
    scoring=scoring,
    refit="RMSE",
    cv=cv,
    n_jobs=-1
)

gs_poly.fit(X_train, y_train)

print("Mejor configuración (Polinomial) — score (neg RMSE):", gs_poly.best_score_)
print("Mejor estimador:", gs_poly.best_estimator_)

In [ ]:
# Tabla resumida de resultados (Top 10 por RMSE)
res_poly = pd.DataFrame(gs_poly.cv_results_).copy()
res_poly["mean_test_RMSE"] = -res_poly["mean_test_RMSE"]
res_poly["mean_test_MAE"]  = -res_poly["mean_test_MAE"]

cols = ["mean_test_RMSE","std_test_RMSE","mean_test_MAE","std_test_MAE","mean_test_R2","std_test_R2","rank_test_RMSE","params"]
display(res_poly[cols].sort_values("mean_test_RMSE").head(10))

In [ ]:
# Evaluación en TEST (solo reporte)
best_poly = gs_poly.best_estimator_
pred_poly_test = best_poly.predict(X_test)
print("TEST metrics (Polinomial):", metrics(y_test, pred_poly_test))

## Actividad 2 — Curva de validación (error vs grado)

In [ ]:
# Para curva de validación: fijamos el escalador (StandardScaler) y variamos degree
degrees = [1,2,3,4,5,6,7]
val_rows = []

for d in degrees:
    model = Pipeline([
        ("preprocess", build_preprocess_with_poly(X_train, scaler_numeric=StandardScaler(), degree=d)),
        ("reg", LinearRegression()),
    ])
    # GridSearchCV de un solo modelo para obtener mean/std CV de RMSE
    gs = GridSearchCV(model, param_grid={}, scoring=scoring, refit="RMSE", cv=cv, n_jobs=-1, return_train_score=True)
    gs.fit(X_train, y_train)
    r = pd.DataFrame(gs.cv_results_).iloc[0]
    val_rows.append({
        "degree": d,
        "train_RMSE_mean": -r["mean_train_RMSE"],
        "cv_RMSE_mean": -r["mean_test_RMSE"],
        "cv_RMSE_std": r["std_test_RMSE"],
    })

val_df = pd.DataFrame(val_rows)
display(val_df)

In [ ]:
# Gráfica train vs CV
x = val_df["degree"].values
train_rmse = val_df["train_RMSE_mean"].values
cv_rmse = val_df["cv_RMSE_mean"].values
cv_std = val_df["cv_RMSE_std"].values

fig = plt.figure()
ax = fig.add_subplot(111)
ax.set_title("Curva de validación — Polinomial (StandardScaler)")
ax.plot(x, train_rmse, marker="o", label="Train mean RMSE")
ax.plot(x, cv_rmse, marker="o", label="CV mean RMSE")
ax.fill_between(x, cv_rmse - cv_std, cv_rmse + cv_std, alpha=0.2, label="±1 std (CV)")
ax.set_xlabel("degree")
ax.set_ylabel("RMSE")
ax.legend()
plt.show()

## Actividad 3 — Regresión regularizada: Ridge y Lasso (GridSearchCV)

In [ ]:
alphas = np.logspace(-4, 3, 25)

# Ridge
ridge_pipe = Pipeline([
    ("preprocess", build_preprocess_no_poly(X_train, scaler_numeric=StandardScaler())),
    ("reg", Ridge(random_state=RANDOM_STATE)),
])

param_grid_ridge = {
    "preprocess": [
        build_preprocess_no_poly(X_train, scaler_numeric=StandardScaler()),
        build_preprocess_no_poly(X_train, scaler_numeric=MinMaxScaler()),
        build_preprocess_no_poly(X_train, scaler_numeric="passthrough"),
    ],
    "reg__alpha": alphas
}

gs_ridge = GridSearchCV(
    ridge_pipe, param_grid=param_grid_ridge, scoring=scoring, refit="RMSE", cv=cv, n_jobs=-1
)
gs_ridge.fit(X_train, y_train)

print("Mejor Ridge — neg RMSE:", gs_ridge.best_score_)
print("Best params:", gs_ridge.best_params_)

In [ ]:
res_ridge = pd.DataFrame(gs_ridge.cv_results_).copy()
res_ridge["mean_test_RMSE"] = -res_ridge["mean_test_RMSE"]
res_ridge["mean_test_MAE"]  = -res_ridge["mean_test_MAE"]

cols = ["mean_test_RMSE","std_test_RMSE","mean_test_MAE","std_test_MAE","mean_test_R2","std_test_R2","rank_test_RMSE","params"]
display(res_ridge[cols].sort_values("mean_test_RMSE").head(10))

best_ridge = gs_ridge.best_estimator_
pred_ridge_test = best_ridge.predict(X_test)
print("TEST metrics (Ridge):", metrics(y_test, pred_ridge_test))

In [ ]:
# Lasso
lasso_pipe = Pipeline([
    ("preprocess", build_preprocess_no_poly(X_train, scaler_numeric=StandardScaler())),
    ("reg", Lasso(max_iter=20000, random_state=RANDOM_STATE)),
])

param_grid_lasso = {
    "preprocess": [
        build_preprocess_no_poly(X_train, scaler_numeric=StandardScaler()),
        build_preprocess_no_poly(X_train, scaler_numeric=MinMaxScaler()),
        build_preprocess_no_poly(X_train, scaler_numeric="passthrough"),
    ],
    "reg__alpha": alphas
}

gs_lasso = GridSearchCV(
    lasso_pipe, param_grid=param_grid_lasso, scoring=scoring, refit="RMSE", cv=cv, n_jobs=-1
)
gs_lasso.fit(X_train, y_train)

print("Mejor Lasso — neg RMSE:", gs_lasso.best_score_)
print("Best params:", gs_lasso.best_params_)

In [ ]:
res_lasso = pd.DataFrame(gs_lasso.cv_results_).copy()
res_lasso["mean_test_RMSE"] = -res_lasso["mean_test_RMSE"]
res_lasso["mean_test_MAE"]  = -res_lasso["mean_test_MAE"]

cols = ["mean_test_RMSE","std_test_RMSE","mean_test_MAE","std_test_MAE","mean_test_R2","std_test_R2","rank_test_RMSE","params"]
display(res_lasso[cols].sort_values("mean_test_RMSE").head(10))

best_lasso = gs_lasso.best_estimator_
pred_lasso_test = best_lasso.predict(X_test)
print("TEST metrics (Lasso):", metrics(y_test, pred_lasso_test))

### Actividad 3 — Variables llevadas a cero por Lasso (selección automática)

In [ ]:
# Para recuperar nombres de features después del preprocesamiento (OneHot), se requiere sklearn>=1.0
# Intentamos extraer; si falla por versión, al menos reportamos coeficientes no-cero por índice.

try:
    pre = best_lasso.named_steps["preprocess"]
    feature_names = pre.get_feature_names_out()
    coefs = best_lasso.named_steps["reg"].coef_
    coef_s = pd.Series(coefs, index=feature_names).sort_values(key=np.abs, ascending=False)
    display(coef_s.head(30))
    print("N coeficientes exactamente cero:", int((coef_s == 0).sum()), "/", len(coef_s))
except Exception as e:
    print("No fue posible extraer nombres de features (versión sklearn). Error:", e)
    coefs = best_lasso.named_steps["reg"].coef_
    print("Coeficientes (primeros 30):", coefs[:30])

## Actividad 4 — Regresión polinomial regularizada (Polinomial + Ridge)

In [ ]:
poly_ridge_pipe = Pipeline([
    ("preprocess", build_preprocess_with_poly(X_train, scaler_numeric=StandardScaler(), degree=2)),
    ("reg", Ridge(random_state=RANDOM_STATE)),
])

degrees = [1,2,3,4,5]
alphas_pr = np.logspace(-4, 3, 15)

preprocess_candidates = []
for sc in [StandardScaler(), MinMaxScaler(), "passthrough"]:
    for d in degrees:
        preprocess_candidates.append(build_preprocess_with_poly(X_train, scaler_numeric=sc, degree=d))

param_grid_poly_ridge = {
    "preprocess": preprocess_candidates,
    "reg__alpha": alphas_pr
}

gs_poly_ridge = GridSearchCV(
    poly_ridge_pipe, param_grid=param_grid_poly_ridge, scoring=scoring, refit="RMSE", cv=cv, n_jobs=-1
)
gs_poly_ridge.fit(X_train, y_train)

print("Mejor Polinomial+Ridge — neg RMSE:", gs_poly_ridge.best_score_)
print("Best params:", gs_poly_ridge.best_params_)

In [ ]:
res_poly_ridge = pd.DataFrame(gs_poly_ridge.cv_results_).copy()
res_poly_ridge["mean_test_RMSE"] = -res_poly_ridge["mean_test_RMSE"]
res_poly_ridge["mean_test_MAE"]  = -res_poly_ridge["mean_test_MAE"]

cols = ["mean_test_RMSE","std_test_RMSE","mean_test_MAE","std_test_MAE","mean_test_R2","std_test_R2","rank_test_RMSE","params"]
display(res_poly_ridge[cols].sort_values("mean_test_RMSE").head(10))

best_poly_ridge = gs_poly_ridge.best_estimator_
pred_poly_ridge_test = best_poly_ridge.predict(X_test)
print("TEST metrics (Polinomial+Ridge):", metrics(y_test, pred_poly_ridge_test))

## Actividad 5 — Comparación y selección del mejor modelo (promedio + estabilidad)

In [ ]:
def best_row(gs: GridSearchCV) -> pd.Series:
    df = pd.DataFrame(gs.cv_results_)
    idx = gs.best_index_
    return df.loc[idx]

def summarize_model(name: str, gs: GridSearchCV) -> dict:
    r = best_row(gs)
    return {
        "Modelo": name,
        "BestParams": gs.best_params_,
        "CV_RMSE_mean": -r["mean_test_RMSE"],
        "CV_RMSE_std": r["std_test_RMSE"],
        "CV_MAE_mean": -r["mean_test_MAE"],
        "CV_MAE_std": r["std_test_MAE"],
        "CV_R2_mean": r["mean_test_R2"],
        "CV_R2_std": r["std_test_R2"],
    }

compare = pd.DataFrame([
    summarize_model("Polinomial", gs_poly),
    summarize_model("Ridge", gs_ridge),
    summarize_model("Lasso", gs_lasso),
    summarize_model("Polinomial+Ridge", gs_poly_ridge),
]).sort_values("CV_RMSE_mean").reset_index(drop=True)

display(compare)

### Escribe aquí tu decisión final (texto)
Criterio sugerido: menor `CV_RMSE_mean` y, si hay diferencias pequeñas, preferir menor `CV_RMSE_std` (más estabilidad).  
Incluye una frase sobre complejidad (grado) y sobre interpretabilidad (Lasso vs Ridge).

## Actividad 6 — Intervalos de confianza (bootstrap en TEST, >= 500 remuestreos)

In [ ]:
def bootstrap_ci(model, X_test: pd.DataFrame, y_test: pd.Series, n_boot: int = 500, ci: float = 0.95):
    rng = np.random.default_rng(RANDOM_STATE)
    n = len(y_test)
    rows = []

    for _ in range(n_boot):
        idx = rng.integers(0, n, size=n)
        Xb = X_test.iloc[idx]
        yb = y_test.iloc[idx]
        pred = model.predict(Xb)
        rows.append(metrics(yb, pred))

    df_samp = pd.DataFrame(rows)
    alpha = (1 - ci) / 2
    out = pd.DataFrame({
        "mean": df_samp.mean(),
        f"ci_{int(ci*100)}_lower": df_samp.quantile(alpha),
        f"ci_{int(ci*100)}_upper": df_samp.quantile(1 - alpha),
    })
    return df_samp, out

def plot_bootstrap(df_samp: pd.DataFrame, bins: int = 30):
    for col in df_samp.columns:
        fig = plt.figure()
        ax = fig.add_subplot(111)
        ax.set_title(f"Bootstrap — {col}")
        ax.hist(df_samp[col].values, bins=bins)
        ax.set_xlabel(col)
        ax.set_ylabel("Frecuencia")
        plt.show()

# Selección automática: tomamos el mejor por CV_RMSE_mean (puedes cambiarlo si argumentas otra cosa)
best_name = compare.iloc[0]["Modelo"]
model_map = {
    "Polinomial": gs_poly.best_estimator_,
    "Ridge": gs_ridge.best_estimator_,
    "Lasso": gs_lasso.best_estimator_,
    "Polinomial+Ridge": gs_poly_ridge.best_estimator_,
}
final_model = model_map[best_name]

# Reentrenar en todo X_train para bootstrap sobre TEST congelado
final_model.fit(X_train, y_train)
test_pred = final_model.predict(X_test)
print("Modelo final:", best_name)
print("Métricas puntuales en TEST:", metrics(y_test, test_pred))

df_samp, df_ci = bootstrap_ci(final_model, X_test, y_test, n_boot=500, ci=0.95)
display(df_ci)
plot_bootstrap(df_samp, bins=30)

## Actividad 7 — Análisis de resultados (responder preguntas del enunciado)

### 7.1 Análisis cuantitativo
Responde con evidencia (tablas/gráficas) y con interpretación:
- ¿Cuál modelo obtuvo el mejor desempeño en test?
- ¿Coincide el mejor desempeño en test con el mejor promedio en validación cruzada? Si no coincide, ¿por qué?
- ¿El modelo con mejor promedio es necesariamente el más adecuado? Discute la desviación estándar.
- Con base en la curva de validación: ¿en qué grado empieza el sobreajuste?
- ¿Cómo afecta la regularización la magnitud y estabilidad de los coeficientes?
- ¿Los IC de bootstrap sugieren estabilidad o alta variabilidad?

### 7.2 Análisis cualitativo
- ¿Qué variables quedaron “seleccionadas” por Lasso (coeficientes no cero)?
- Interpretación práctica de coeficientes del modelo final.
- ¿Hay diferencias entre el modelo más preciso y el más interpretable?
- ¿Qué decisiones estratégicas podría tomar AlpesHearth con estos resultados?

### 7.3 Reflexión conceptual
- Relación complejidad–generalización–estabilidad.
- Fuentes de sesgo potenciales (datos + modelado).
- Si el tamaño de muestra fuera mayor: ¿esperas cambios en estabilidad y por qué?

## Sección obligatoria — Uso de herramientas de IA generativa

Incluye:
1) herramienta y para qué la usaste,  
2) prompts principales,  
3) evaluación crítica (errores/limitaciones/correcciones),  
4) aportes propios (qué decidiste tú y por qué).

## (Opcional) Exportar predicciones para el archivo de test (si el curso lo requiere)

In [ ]:
# Si necesitas generar un CSV para el test original (mismo formato de separador), usa esto.
# Nota: esto NO es parte explícita del Lab 2, pero lo dejamos por consistencia con Lab 1.

# 1) Entrenar en todo el train limpio (no split) con el modelo final seleccionado
X_full = train.drop(columns=[TARGET] + DROP_ALWAYS, errors="ignore").copy()
y_full = train[TARGET].copy()

final_model.fit(X_full, y_full)

# 2) Preparar X_submit con columnas alineadas
test_submit = clean_common(test_raw.copy(), CAT_COLS)
X_submit = test_submit.drop(columns=DROP_ALWAYS, errors="ignore").copy()
X_submit = X_submit.reindex(columns=X_full.columns)

# 3) Predecir y exportar
out = test_raw.copy()
out[TARGET] = final_model.predict(X_submit)

output_path = "Datos_Test_Lab2_con_pred.csv"
out.to_csv(output_path, index=False, sep=";")
print("Archivo exportado:", output_path)
display(out.head())